# Langchain을 활용하지 않은 RAG

# 1. python doxc 활용하여 문서 조회

In [ ]:
%pip install python-docx

In [ ]:
import sys
print(sys.executable)

import docx
print(docx.__file__)

In [ ]:
from docx import Document
document = Document('./data/tax.docx')

full_text = ''
for index, paragraph in enumerate(document.paragraphs) :
    #print(f'paragraph == {paragraph.text}')
    full_text += f'{paragraph.text}\n'

# 2. 문서 분할
openai tiktoken 활용
- tiktoken으로 문서를 인코딩한다
- 인코딩한 결과를 지정한 청크 사이즈 만큼 분할한다.
- 분할한 청크를 디코딩해서 list에 저장한다.
    - 이유 : 컨텍스트 윈도우를 넘어가기 때문에 토큰 절약을 위해

In [ ]:
%pip install tiktoken

In [ ]:
import tiktoken

def split_text(full_text, chunk_size):
    encorder = tiktoken.encoding_for_model("gpt-4o")
    total_encoding = encorder.encode(full_text)
    total_token_count = len(total_encoding)

    text_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i: i + chunk_size]
        decoded = encorder.decode(chunk)

        text_list.append(decoded)

    return text_list


In [ ]:
chunk_list = split_text(full_text, 1500)

chunk_list

# 3. 임베딩

In [ ]:
%pip install chromadb

In [ ]:
import chromadb

chroma_client = chromadb.Client()

In [ ]:
# collection : RDBMS에서 테이블과 같은 개념
collection_name = 'tex_collection'
tax_collection = chroma_client.create_collection(collection_name)

In [ ]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from dotenv import load_dotenv

load_dotenv()

openai_embedding = OpenAIEmbeddingFunction(model_name='text-embedding-3-large')

In [ ]:
chroma_client.delete_collection(collection_name)

tax_collection = chroma_client.get_or_create_collection(collection_name, embedding_function=openai_embedding)

In [ ]:
id_list = []
for index in range(len(chunk_list)):
    id_list.append(f'{index}')

tax_collection.add(documents=chunk_list, ids=id_list)

# 4. 유사도 검색

In [ ]:
query = '연봉 5000만원 직장인의 소득세는 얼마인가요?'

retrieved_doc = tax_collection.query(query_texts=query, n_results=3)

retrieved_doc['documents'][0]

# 5. 질의

In [ ]:
%pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": f"당신은 한국의 소득세 전문가입니다. 아래 내용을 참고해서 사용자의 질문에 답변해주세요 {retrieved_doc['documents'][0]}"},
        {"role": "user", "content":query}
    ]
)


In [59]:
response.choices[0].message.content

'연봉 5,000만원에 대한 소득세를 계산하기 위해서는 소득공제와 세액공제를 먼저 고려해야 합니다. 일반적으로 소득세는 근로소득 공제, 기본공제와 추가적인 세액공제를 고려하여 계산됩니다. 아래는 대략적인 계산 과정입니다:\n\n1. **근로소득 공제**: \n   - 연봉 5,000만원의 경우, 근로소득 공제는 다음과 같이 계산됩니다:\n     \\[\n     근로소득 공제액 = 1,200만원 + (총급여액 - 4,500만원) \\times 40\\%\n     \\]\n     \\[\n     = 1,200만원 + (5,000만원 - 4,500만원) \\times 40\\% = 1,200만원 + 200만원 = 1,400만원\n     \\]\n\n2. **과세표준 계산**:\n   \\[\n   과세표준 = 연봉 - 근로소득 공제\n   \\]\n   \\[\n   = 5,000만원 - 1,400만원 = 3,600만원\n   \\]\n\n3. **소득세 계산**:\n   - 소득세율은 기본세율을 적용합니다. 과세표준 3,600만원에 대한 세율은 다음과 같습니다:\n     - 1,200만원 이하: 6%\n     - 1,200만원 초과 4,600만원 이하: 15%\n   \n   - 따라서:\n     \\[\n     1,200만원 \\times 6\\% + (3,600만원 - 1,200만원) \\times 15\\%\n     \\]\n     \\[\n     = 72만원 + 2,400만원 \\times 15\\% = 72만원 + 360만원 = 432만원\n     \\]\n\n4. **세액공제 적용** (해당 공제를 모두 포함한 가정):\n   - 예를 들어, 공제 항목에 따라 세액공제가 있을 수 있습니다. 구체적인 세액공제 항목은 개인에 따라 달라질 수 있음을 유념해야 합니다. 예를 들어, 인적공제, 특별세액공제 등이 있을 수 있으며, 각 항목에 대해 공제 내용을 반영하여야 합니다.\n\n위의 계산은 대략적인 소득세 계산 방식이며, 실제로는 개